# 05 · RNN: reading the heartbeat as a sequence

The CNN (notebook 04) looks at each 3 s window as an **image**. A recurrent network reads the same window as a
**sequence**: one spectrogram column at a time, left to right, keeping a memory of what it has heard. Heart sounds
are a sequence by nature (S1, systole, S2, diastole, again), and a murmur is defined by *where* it sits in that
sequence (mostly in systole, between S1 and S2), so order should matter.

To make the comparison fair, **only the network changes**: same input (the 3-channel spectrogram: full, beat-locked
part, leftover), same folds, same loss weighted 5/3/1, same early stopping, same 3 seeds, same patient rule.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from murmur.core.constants import CLASSES
from murmur.core.metric import confusion, weighted_accuracy
from murmur.core.paths import results_dir
from murmur.evaluate import decisions
from murmur.models import patient
from murmur.models.rnn import SmallGRU
from murmur.models.cnn import SmallCNN
from murmur.train import cv, cv_rnn

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
RECOMPUTE = False
SEEDS = [0, 1, 2]
cnn_dir = lambda seed: results_dir() / "cnn" / "3-channel" / f"seed{seed}"
gru_dir = lambda seed: cv_rnn.run_dir("3-channel", seed)

train, patients, train_recs, patient_of, _ = cv.training_set()

## Architecture

A window is `(3 channels, 50 frequency bins, 300 frames)`. The GRU reads it as **300 steps of 150 numbers**
(3 × 50, one column of every channel).

```
300 steps × 150  →  linear 150→64 + LayerNorm + ReLU
                 →  GRU, 2 layers, 64 units, bidirectional  →  300 × 128
                 →  mean and max over time                  →  256
                 →  dropout 0.3 → linear → 3 logits
```

- **GRU** rather than a plain RNN: its gates keep information over hundreds of steps (a plain RNN forgets);
  rather than an LSTM: similar results with fewer weights, and we have ~750 patients.
- **Bidirectional**: at each moment the network knows what came before *and* after; to tell systole from
  diastole you need to know where S1 and S2 are.
- **Mean + max pooling**: the mean summarises the whole window (noise → Unknown); the max catches the strongest
  moment (a murmur in some beats only).
- **Gradient clipping** (±1 per element): recurrent networks can have exploding gradients.

In [ ]:
n_weights = lambda m: sum(p.numel() for p in m.parameters())
gru = SmallGRU(n_bins=50, in_channels=3)
print(f"GRU {n_weights(gru):,} weights, CNN {n_weights(SmallCNN(in_channels=3)):,} weights")
print("one window in → logits out:", tuple(gru(torch.zeros(2, 3, 50, 300)).shape))
if RECOMPUTE:                      # ~15 trainings (5 folds × 3 seeds); a GPU helps a lot
    for seed in SEEDS:
        cv_rnn.run(gru_dir(seed), "repet", repeat=seed)

## Training curves (inner validation, seed 0)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for fold in range(5):
    h = pd.read_csv(gru_dir(0) / f"history-fold{fold}.tsv", sep="\t")
    ax.plot(h["epoch"], h["val_wa"], label=f"fold {fold}")
    best = h.loc[h["val_wa"].idxmax()]
    ax.plot(best["epoch"], best["val_wa"], "k.", ms=8)
ax.set(xlabel="epoch", ylabel="inner validation WA", title="GRU, 3-channel, seed 0")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()
pd.DataFrame([pd.read_json(gru_dir(s) / "summary.json")[["best_epoch", "epochs_run", "seconds"]].mean()
              .rename(f"seed {s}") for s in SEEDS]).round(1)

## GRU vs CNN

Out-of-fold weighted accuracy over the 753 training patients (the test split is not used): each seed, each fold
(mean over seeds), and the 3 seeds' predictions averaged.

In [ ]:
probs = {"CNN": decisions.seeds([cnn_dir(s) for s in SEEDS], patient_of),
         "GRU": decisions.seeds([gru_dir(s) for s in SEEDS], patient_of)}
idx = probs["CNN"][0].index
y, folds = patients.loc[idx, "murmur"], train.loc[idx, "fold"]
per_fold = {m: np.mean([decisions.fold_wa(p, y, folds) for p in probs[m]], axis=0) for m in probs}
table = pd.DataFrame({m: {**{f"seed {s}": weighted_accuracy(y, patient.decide(p)) for s, p in zip(SEEDS, probs[m])},
                          "mean ± sd over seeds": "{:.3f} ± {:.3f}".format(*(lambda w: (np.mean(w), np.std(w)))(
                              [weighted_accuracy(y, patient.decide(p)) for p in probs[m]])),
                          **{f"fold {k} (seed mean)": w for k, w in enumerate(per_fold[m])},
                          "seed ensemble": weighted_accuracy(y, patient.decide(decisions.ensemble(probs[m])))}
                      for m in probs})
print(f"GRU wins {int((per_fold['GRU'] > per_fold['CNN']).sum())} of 5 folds, "
      f"mean difference {np.mean(per_fold['GRU'] - per_fold['CNN']):+.3f}")
table.round(3)

## Patient-level confusion matrices (3 seeds averaged)

In [ ]:
for m in probs:
    pred = patient.decide(decisions.ensemble(probs[m]))
    display(pd.DataFrame(confusion(y, pred), index=[f"true {c}" for c in CLASSES],
                         columns=[f"pred {c}" for c in CLASSES]).rename_axis(m))

## Do they make the same mistakes?

If the two networks miss different patients, averaging them could beat both.

In [ ]:
ens = {m: decisions.ensemble(probs[m]) for m in probs}
both = (ens["CNN"] + ens["GRU"]) / 2
print("CNN + GRU averaged:", round(weighted_accuracy(y, patient.decide(both)), 3))
display(pd.DataFrame(confusion(y, patient.decide(both)), index=[f"true {c}" for c in CLASSES],
                     columns=[f"pred {c}" for c in CLASSES]).rename_axis("CNN + GRU"))

## Conclusion

_To fill in once the runs are done._